<a href="https://colab.research.google.com/github/nnekaanadu-art/contact-centre-agent-assist/blob/main/agent_assist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Contact Centre Agent Assist | AI Proof of Concept

**Stage:** Build & Validate · **Data:** 20 synthetic Northbridge Bank call transcripts · **Prompt:** Version 2

This notebook generates five recommendations for each completed transcript and evaluates four structured fields against a synthetic answer key. It does **not** connect to a real bank, save customer notes, or automate decisions. Human review is required for any operational use.

### How to run
1. In Colab, add a secret named `GEMINI_API_KEY` (key icon on the left), and enable notebook access. Never paste the key into a code cell.
2. Run the setup cells. The notebook automatically loads the synthetic transcripts and answer key from the published `data/results_v2.csv` file on GitHub.
3. Run the analysis and scoring cells in order. The published results are used only as input data; a fresh run generates new outputs. Model access and quotas may change; API calls can incur costs depending on your account.
4. Download the generated results CSV if desired. The existing published Version 1/Version 2 results are historical runs; reruns may differ.

**Evaluation caveat:** The 20-transcript dataset was used during prompt iteration; it is not an independent holdout. The summary scores (4.35 and 4.50) were assessed separately and are not recomputed here.


In [ ]:
!pip install -q google-genai pandas


In [ ]:
from google.colab import userdata, files
from google import genai
from google.genai import types
import pandas as pd
import json, time

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets and enable notebook access.")
client = genai.Client(api_key=api_key)
# Keep this aligned with the model used for a recorded evaluation.
MODEL = "gemini-3.5-flash-lite"
PROMPT_VERSION = "v2"


## Load synthetic evaluation data from GitHub

The published `data/results_v2.csv` includes the synthetic transcripts, answer key, and historical model outputs. This notebook loads **only** the transcript and expected-answer columns, so historical predictions cannot be mistaken for fresh outputs. No file upload is required. The published results files are not modified.


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/nnekaanadu-art/contact-centre-agent-assist/main/data/results_v2.csv"
INPUT_COLUMNS = ["transcript_id", "transcript", "expected_intent", "expected_complaint", "expected_vulnerability", "expected_next_best_action"]
try:
    df = pd.read_csv(DATA_URL, usecols=INPUT_COLUMNS)
except Exception as exc:
    raise RuntimeError("Could not load the published synthetic dataset from GitHub. Check the URL and your network connection.") from exc
required_input = set(INPUT_COLUMNS)
missing = required_input - set(df.columns)
if missing:
    raise ValueError(f"Missing input columns: {sorted(missing)}")
if df["transcript_id"].duplicated().any():
    raise ValueError("Transcript IDs must be unique.")
if df[INPUT_COLUMNS].isna().any().any():
    raise ValueError("Input data contains missing values.")
print(f"Loaded {len(df)} synthetic transcripts from GitHub.")
df[["transcript_id", "expected_intent", "expected_complaint", "expected_vulnerability"]].head()


In [ ]:
PROMPT_VERSION = "v2"
SYSTEM_PROMPT = """
You are an assistant for contact centre agents at a Canadian bank.
Read the call transcript and return only a JSON object with these keys:

"summary": three sentences describing the reason for the call, what happened, and what is still open.
"intent": one of lost_stolen_card, transaction_dispute, suspected_fraud, fee_reversal, online_banking_access, mortgage_payment_change, address_update, credit_limit_increase, account_closure, bereavement, balance_inquiry.
"complaint": "Yes" or "No".
"vulnerability": "Yes" or "No".
"next_best_action": one of "Block card and order replacement", "Open transaction dispute case", "Escalate to fraud team", "Apply fee reversal per policy", "Reset online banking access", "Transfer to mortgage specialist", "Update address after verification", "Submit credit limit request", "Offer retention review before closure", "Refer to bereavement support team", "Provide information only".

Definitions:
"complaint" is "Yes" when the customer expresses dissatisfaction with the bank itself (its fees, service, or errors) or asks for their concern to be recorded. Frustration with a merchant alone is not a complaint against the bank.
"vulnerability" is "Yes" when the customer mentions something that may reduce their ability to manage their finances: bereavement, illness, age related confusion, job loss, or being pressured or manipulated by a scammer in a way that affects their ability to protect themselves. A customer who spots fraud or theft and reports it promptly is not vulnerable on that basis alone.

Return JSON only, with no other text.
"""

## Generate structured outputs

The model is asked to return JSON. The notebook also checks required fields and approved labels, retries transient failures, and records any unresolved errors. Failed calls are **not** counted as correct.


In [ ]:
INTENTS = {"lost_stolen_card", "transaction_dispute", "suspected_fraud", "fee_reversal", "online_banking_access", "mortgage_payment_change", "address_update", "credit_limit_increase", "account_closure", "bereavement", "balance_inquiry"}
ACTIONS = {"Block card and order replacement", "Open transaction dispute case", "Escalate to fraud team", "Apply fee reversal per policy", "Reset online banking access", "Transfer to mortgage specialist", "Update address after verification", "Submit credit limit request", "Offer retention review before closure", "Refer to bereavement support team", "Provide information only"}
REQUIRED_OUTPUT = {"summary", "intent", "complaint", "vulnerability", "next_best_action"}

def validate_output(output):
    if not isinstance(output, dict) or not REQUIRED_OUTPUT.issubset(output):
        raise ValueError("Response missing one or more required fields")
    if not isinstance(output["summary"], str) or not output["summary"].strip():
        raise ValueError("Empty or invalid summary")
    if output["intent"] not in INTENTS or output["next_best_action"] not in ACTIONS:
        raise ValueError("Unapproved intent or action")
    if output["complaint"] not in {"Yes", "No"} or output["vulnerability"] not in {"Yes", "No"}:
        raise ValueError("Flags must be Yes or No")
    return output

def analyze(transcript):
    response = client.models.generate_content(
        model=MODEL, contents=transcript,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT, response_mime_type="application/json",
            http_options=types.HttpOptions(timeout=60000),
        ),
    )
    return validate_output(json.loads(response.text))

def analyze_with_retry(transcript, attempts=2):
    for attempt in range(attempts):
        try:
            return analyze(transcript)
        except Exception as exc:
            if attempt == attempts - 1:
                return {"error": f"{type(exc).__name__}: {str(exc)[:300]}"}
            time.sleep(15)

results = []
for _, row in df.iterrows():
    output = analyze_with_retry(row["transcript"])
    output["transcript_id"] = row["transcript_id"]
    results.append(output)
    print(row["transcript_id"], "ERROR" if "error" in output else "OK")
    time.sleep(15)  # pacing for constrained API quotas
results_df = pd.DataFrame(results)
print(f"Successful: {sum('error' not in r for r in results)}/{len(results)}")


## Evaluate final results

The four accuracy measures compare the model output to the provided answer key. Incomplete/invalid responses count as incorrect. Accuracy does not measure recall or precision; those are additional proposed PRD measures.


In [ ]:
FIELDS = {"intent": "expected_intent", "complaint": "expected_complaint", "vulnerability": "expected_vulnerability", "next_best_action": "expected_next_best_action"}
scored = df.merge(results_df, on="transcript_id", how="left", validate="one_to_one")
for model_col, key_col in FIELDS.items():
    if model_col not in scored:
        scored[model_col] = None
    scored[model_col + "_correct"] = (
        scored[model_col].fillna("").astype(str).str.strip().str.lower()
        == scored[key_col].fillna("").astype(str).str.strip().str.lower()
    )
print("Prompt version:", PROMPT_VERSION, "| Model:", MODEL)
for field in FIELDS:
    print(f"{field}: {scored[field + '_correct'].mean() * 100:.0f}%")
print(f"All four fields correct: {scored[[c + '_correct' for c in FIELDS]].all(axis=1).mean() * 100:.0f}%")
print("Note: Rerun results may differ from historical README figures.")


In [ ]:
for field, expected in FIELDS.items():
    misses = scored.loc[~scored[field + "_correct"]]
    for _, row in misses.iterrows():
        print(f"{row['transcript_id']} | {field} | expected: {row[expected]} | got: {row[field]}")


In [ ]:
output_name = f"results_{PROMPT_VERSION}_rerun.csv"
scored.to_csv(output_name, index=False)
files.download(output_name)


## Interpretation and next steps

The published Version 2 evaluation identified a false positive on transcript **T03** for vulnerability classification. Prompt changes should be assessed on the full dataset, then on an unseen holdout set. Before any operational pilot, also evaluate summary factuality, repeated-run stability, recall/precision for both flags, privacy, failure behavior, and human-review workflows.

[Portfolio Home](https://nnekaanadu-art.github.io/) · [LinkedIn](https://www.linkedin.com/in/NnekaAnadu) · [GitHub](https://github.com/nnekaanadu-art)

© 2026 Nneka Anadu. All rights reserved.
